# Model Comparison — Document Classifier Benchmark

**Purpose**: Benchmark 5 classical ML classifiers on our 4-class document dataset to justify the choice of LinearSVC for the production pipeline.

**Models compared**:
1. Multinomial Naive Bayes — classical baseline for text
2. Logistic Regression — linear model with probabilistic output
3. LinearSVC — current production choice
4. Random Forest — non-linear, tree-based
5. Voting Classifier (soft) — ensemble of the above

**Evaluation**: 5-fold cross-validation on the training set + held-out test set metrics.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from scipy.sparse import hstack

ROOT = Path().resolve().parent
DATA_PROC = ROOT / "data" / "processed"

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Setup complete.")

## 1. Load data

In [ ]:
train = pd.read_csv(DATA_PROC / "train.csv").dropna(subset=["text", "label"])
test  = pd.read_csv(DATA_PROC / "test.csv").dropna(subset=["text", "label"])

LABELS = ["invoice", "email", "scientific_report", "letter"]
train = train[train["label"].isin(LABELS)]
test  = test[test["label"].isin(LABELS)]

print(f"Train: {len(train):,}  |  Test: {len(test):,}")
print(f"\nTrain distribution:\n{train['label'].value_counts()}")
print(f"\nTest distribution:\n{test['label'].value_counts()}")

X_train = train["text"].tolist()
y_train = train["label"].tolist()
X_test  = test["text"].tolist()
y_test  = test["label"].tolist()

## 2. Feature engineering (same as production pipeline)

We build the identical TF-IDF features as `src/train.py`: word 1-2 grams + char 3-5 grams, stacked horizontally.

In [ ]:
word_vec = TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2),
    min_df=2, max_df=0.95, max_features=80_000,
    sublinear_tf=True, strip_accents="unicode",
    token_pattern=r"(?u)\b\w\w+\b",
)
char_vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5),
    min_df=3, max_df=0.95, max_features=60_000,
    sublinear_tf=True, strip_accents="unicode",
)

X_train_word = word_vec.fit_transform(X_train)
X_test_word  = word_vec.transform(X_test)
X_train_char = char_vec.fit_transform(X_train)
X_test_char  = char_vec.transform(X_test)

X_train_feat = hstack([X_train_word, X_train_char]).tocsr()
X_test_feat  = hstack([X_test_word,  X_test_char]).tocsr()

print(f"Train feature matrix: {X_train_feat.shape}")
print(f"Test  feature matrix: {X_test_feat.shape}")

## 3. Define candidate models

Note on **RandomForest**: trees don't work well with 140k-dimensional sparse TF-IDF directly. We pass the same features but expect it to underperform — this is itself a useful finding.

Note on **VotingClassifier (soft)**: requires `predict_proba`. LinearSVC doesn't natively expose it, so we wrap it in `CalibratedClassifierCV` (Platt scaling).

In [ ]:
models = {
    "MultinomialNB":       MultinomialNB(),
    "LogisticRegression":  LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42),
    "LinearSVC":           LinearSVC(C=1.0, max_iter=2000, random_state=42),
    "RandomForest":        RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
}

# Soft-voting ensemble needs probability-capable estimators
voting = VotingClassifier(
    estimators=[
        ("nb",  MultinomialNB()),
        ("lr",  LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)),
        ("svc", CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42), cv=3)),
    ],
    voting="soft",
    n_jobs=-1,
)
models["VotingEnsemble"] = voting

print(f"Defined {len(models)} models: {list(models.keys())}")

## 4. 5-fold cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in models.items():
    print(f"▶ {name:20s} …", end=" ", flush=True)
    scores = cross_val_score(clf, X_train_feat, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_results[name] = scores
    print(f"{scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")

cv_df = pd.DataFrame(cv_results)
cv_summary = pd.DataFrame({
    "mean_accuracy": cv_df.mean(),
    "std":           cv_df.std(),
}).sort_values("mean_accuracy", ascending=False)
cv_summary

## 5. Fit on full train, evaluate on held-out test

In [ ]:
test_results = []

for name, clf in models.items():
    print(f"▶ Training {name}…", end=" ", flush=True)
    clf.fit(X_train_feat, y_train)
    y_pred = clf.predict(X_test_feat)
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro")
    test_results.append({"model": name, "test_accuracy": acc, "test_macro_f1": f1})
    print(f"acc={acc*100:.2f}%  f1={f1:.4f}")

test_df = pd.DataFrame(test_results).set_index("model").sort_values("test_accuracy", ascending=False)
test_df

## 6. Visualisations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cv_df_melted = cv_df.melt(var_name="model", value_name="accuracy")
sns.boxplot(data=cv_df_melted, x="model", y="accuracy", ax=ax, palette="Blues")
ax.set_title("5-Fold CV Accuracy per Model")
ax.set_ylabel("Accuracy")
ax.set_xlabel("")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "cv_boxplot.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
(ROOT / "reports").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
test_df["test_accuracy"].plot.barh(ax=ax, color="#2563eb")
ax.set_title("Held-out Test Accuracy per Model")
ax.set_xlabel("Accuracy")
ax.set_xlim(0.9, 1.0)
for i, v in enumerate(test_df["test_accuracy"]):
    ax.text(v + 0.001, i, f"{v*100:.2f}%", va="center")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "test_accuracy.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
best_name = test_df.index[0]
best_model = models[best_name]
y_pred = best_model.predict(X_test_feat)
cm = confusion_matrix(y_test, y_pred, labels=LABELS)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title(f"Confusion Matrix — {best_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\n{best_name} classification report:\n")
print(classification_report(y_test, y_pred, target_names=LABELS))